In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.drop_duplicates(inplace=True)

In [4]:
df.shape

(49582, 2)

## Pre-Processing

1. ## Converting to lowercase

In [5]:
df["review"] = df["review"].str.lower()

## 2. Removing the URLs

In [6]:
import re

## sample_text = "abc is the word, abc => xyz
## new_text = re.sub("abc", "xyz", sample_text)


In [7]:
def remove_urls(text):
    text = re.sub(r"http\S+", "", text)
    return text

df["review"] = df["review"].apply(remove_urls)

### 3. Removing punctuations

In [8]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text) # A-Z , a-z, 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

## 4. Removing HTML

In [9]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

## 5. Removing the Stopwords

In [10]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [12]:
## sample_text = "the love coding in python"
## tokens = word_tokenize(sample_text)

In [13]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    ## now we have to remove stopwords from new tokens
    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text
df["review"] = df["review"].apply(remove_stopwords)

### 6. Stemming

In [14]:
## running -- run
## played -- play
## POterstemming

from nltk.stem import PorterStemmer

In [15]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

### 7. Encoding

In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [17]:
y = df["sentiment"]

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

### Dataset and DataLoader

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [20]:
X_train.shape

(39665, 5000)

In [21]:
X_test.shape

(9917, 5000)

In [22]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [23]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [24]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [25]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN

In [28]:
import torch.nn as nn
import torch.optim as optim 

In [32]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        ## RNN Layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        ## fully connected Layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional ==> (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden step of all the timestamps => (batch, seq_len, hiddne_size)
        # 2nd value = final hidden state of last timestamp

        out = self.fc(out[:, -1, :])
        return out
        

In [33]:
input_size = X_train.shape[1]

model =  RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [35]:
# unsqueeze squeeze

epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) #add singleton direction

        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # batch size => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weight update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.31046783924102783
epoch = 2/10 and loss = 0.1993618607521057
epoch = 3/10 and loss = 0.2331395447254181
epoch = 4/10 and loss = 0.14819130301475525
epoch = 5/10 and loss = 0.18902350962162018
epoch = 6/10 and loss = 0.21291491389274597
epoch = 7/10 and loss = 0.22126521170139313
epoch = 8/10 and loss = 0.25462785363197327
epoch = 9/10 and loss = 0.28247424960136414
epoch = 10/10 and loss = 0.3129577338695526


In [36]:
## evalute

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy= {correct_vals/tot_vals*100}")
        

accuracy= 85.60048401734396
